# Days with < 288 samples
This notebook investigates why we've seen small daily sample counts in Flair.

Lane found that in Flair, there are many more days with <288 samples. He asks why. Let's take a look and check if this is already present in the raw data.

In [ ]:
import os, random
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from matplotlib import pyplot as plt
current_dir = os.getcwd(); 
import sys
sys.path.append('../..')
from src import drawing
from src import pandas_helper
from src import cdf

In [ ]:
def parse_flair_dates(dates):
    """Parse date strings separately for those with/without time component, interpret those without as midnight (00AM)
        Args:
            df (pandas DataFrame): data frame holding data
            date_column (string): column name that holds date time strings to be used for parsing

        Returns:
            pandas series: with parsed dates
        """
    #make sure to only parse dates if the value is not null
    only_date = dates.apply(len) <=10
    dates_copy = dates.copy()
    dates_copy.loc[only_date] = pd.to_datetime(dates.loc[only_date], format='%m/%d/%Y')
    dates_copy.loc[~only_date] = pd.to_datetime(dates.loc[~only_date], format='%m/%d/%Y %I:%M:%S %p')
    return dates_copy

def load_data(file_name, columns=None):
    path = os.path.join(current_dir, '..', '..', 'data' ,'raw', 'FLAIRPublicDataSet.zip', 'Data Tables', file_name)
    kwargs = {"usecols": columns} if columns else {}
    df = pandas_helper.get_df(path, **kwargs)
    #df = pd.read_csv(path, sep="|", low_memory=False, **kwargs)
    df['DateTime'] = df.loc[df.DataDtTm.notna(),'DataDtTm'].transform(parse_flair_dates)
    df['DateTimeAdjusted'] = df.loc[df.DataDtTm_adjusted.notna(),'DataDtTm_adjusted'].transform(parse_flair_dates)
    return df

df_cgm = load_data('FLAIRDeviceCGM.txt')#, columns=['PtID', 'DataDtTm', 'DataDtTm_adjusted', 'CGM'])
df_insulin = load_data('FLAIRDevicePump.txt', columns=['RecID','PtID', 'DataDtTm', 'NewDeviceDtTm', 'DataDtTm_adjusted', 'BasalRt', 
                                                       'TempBasalAmt','TempBasalType', 'TempBasalDur','BolusType', 
                                                       'BolusSource', 'BolusDeliv', 'BolusSelected', 'ExtendBolusDuration', 'BasalRtUnKnown', 
                                                       'Suspend','PrimeVolumeDeliv','Rewind'])
display(df_cgm.head(2))
display(df_cgm.loc[df_cgm.DataDtTm_adjusted.notna()].head(2))
display(df_insulin.head(2))

Let's take a look at the distribution and compare to what Lane found

In [ ]:
df_cgm['day'] = df_cgm.DateTime.apply(lambda x: x.date())
cgm_daily_counts = df_cgm.groupby(['PtID', 'day']).size().reset_index(name='count')
cdf.plot_cdf(cgm_daily_counts['count'], title='CGM Daily Counts', xlabel='Number of CGM readings per day', ylabel='CDF')

sample_availability = cgm_daily_counts['count'].clip(upper=288)/288
print(f'{sample_availability.mean():.2%} of CGM samples are available')

Yes, this matches what lane found, let's take a closer look why this might be

### Could it be because at study start/end there are  more data points missing?

In [ ]:
plt.figure(figsize=(8, 3)); ax=plt.gca()
cgm_daily_counts['day_since_start'] = (cgm_daily_counts['day'] - cgm_daily_counts['day'].min())
cgm_daily_counts['day_since_start'] = pd.to_timedelta(cgm_daily_counts['day_since_start']).dt.days

#missing samples per day
cgm_daily_counts['missing'] = 288-cgm_daily_counts['count'].clip(upper=288)
r = cgm_daily_counts.groupby('day_since_start').agg({'missing': 'sum','PtID': 'nunique'}).reset_index()
ax.scatter(r['day_since_start'], r['missing'], marker='o', s=10, label='Missing CGM readings',alpha=0.5)
twinx = ax.twinx()
twinx.plot(r['day_since_start'], r['PtID'], linestyle='-', color='orange', markersize=3, label='Patients per day')

twintwinax = ax.twinx()
twintwinax.spines['right'].set_position(('outward', 60))
twintwinax.scatter(r['day_since_start'], r['missing']/r['PtID'],  s=3, color='red',label='Missing CGM readings/Patient')
ax.legend(loc='upper left')
twinx.legend(loc='upper right')
twintwinax.legend(loc='lower right')


Overall, most samples are missing mid study. At the beginning/end there are just fewer patients causing some higher misses/day. TO understand better if start/end times causes more outliers we would need to know exactly when patients started/ended. Right now we assume all patients tarted at the same time.

Let's go back and start looking at some examples:

In [ ]:
sample = cgm_daily_counts.loc[cgm_daily_counts['count'] < 200].sample(1).iloc[0]
day = sample['day']
day_start = datetime.combine(day, datetime.min.time())
day_end = day_start + timedelta(days=1)
sub_frame = df_cgm.loc[(df_cgm['PtID'] == sample['PtID']) & (df_cgm['day'] == sample['day'])].copy()

plt.figure(figsize=(8, 3)); ax=plt.gca()
drawing.drawCGM(ax,sub_frame.DateTime, sub_frame.CGM)

plt.title(f'CGM readings for {sample["PtID"]} on {sample["day"]}'); plt.xlabel('Time'); plt.ylabel('CGM (mg/dL)')
plt.xlim(day_start, day_end)
drawing.format_time_axis(ax)

plt.tight_layout()

It appears as if the gaps are rather long, let's take a look at the distribution:

In [ ]:
cgm_gaps = df_cgm.sort_values('DateTime').groupby('PtID')['DateTime'].diff()
cgm_gaps=cgm_gaps.dropna()
cgm_gaps = pd.to_timedelta(cgm_gaps)

In [ ]:
plt.figure(figsize=(7, 2));ax=plt.gca()
cdf.plot_cdf(cgm_gaps[cgm_gaps>timedelta(minutes=15)].dt.total_seconds()/3600, ax=ax)
plt.xscale('log')
ax.set_title('Flair CGM Gaps (>15 min)')
plt.xlabel('Gap duration (hours)')
plt.ylabel('CDF')

plt.scatter(2.1, 0.52, s=300, facecolors='none', edgecolors='red', linewidths=1, zorder=10,label='sensor warmup?')
plt.legend(loc='lower right')
plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/flair_cgm_gaps.png'), bbox_inches='tight', dpi=300)


In [ ]:
num_patients = df_cgm.PtID.nunique()
print(f'Number of patients with CGM data: {num_patients}')
r = df_cgm.groupby('PtID').DateTime.apply(lambda x: x.max() - x.min())
print(f'Average CGM data duration: {r.mean().days} days')

1. We know that the Guardian Sensor 3 has 2 hour warmup time and a maximum duration of 7 days. 
2. The study has ~ 113 patients with a average wear duration of 204 days. 

So we can expect the number of sensor changes to be approximately **113*204/7 = 3293**


In [ ]:
print(f'There are {len(cgm_gaps.loc[cgm_gaps > timedelta(hours=2)])} CGM gaps longer than 2 hours')
print(f'Approximately 3293 of these {100*3293/len(cgm_gaps.loc[cgm_gaps > timedelta(hours=2)]):.1f}% can be explained by sensor warmup')


Summary:   
 - The many days with <288 samples also exist in the raw data
 - The gaps are smoothly distributed with a small spike at 2 hours likely sensor warmup
 - Approximately 1/3 of all gaps >=2 hours can likely be explained as a result of warmup 
 - Many gaps 2-10 hours, unclear why

 Further investigation could look try clipping the data before/after study start/end using the `FLAIRVisitInfo.txt`
